# Lecture 6 - The Haunted DAG

McElreath's lectures for the whole book are available here: https://github.com/rmcelreath/stat_rethinking_2022

An R/Stan repo of code is available here: https://vincentarelbundock.github.io/rethinking2/

An excellent port to Python/PyMC Code is available here: https://github.com/dustinstansbury/statistical-rethinking-2023

You are encouraged to work through both of these versions to re-enforce what we're doing in class.

In [ ]:
# Load R packages
library(cmdstanr)    # R interface to Stan
library(posterior)   # Working with posterior draws
library(bayesplot)   # Plotting posterior draws
library(ggplot2)     # bayesplot figures are ggplots, so we can add to them

stdize <- function(x) (x - mean(x))/sd(x)

# Save the current figure to file (uncomment the savefig() calls below to use)
savefig <- function(file, width = 7, height = 5){
    dev.copy(jpeg, file, width = width, height = height, units = "in", res = 300)
    invisible(dev.off())
}

# Stan's sampler with the same defaults as pm.sample(1000): 4 chains, 1000 warmup, 1000 draws
sample_stan <- function(model, data){
    model$sample(data = data, chains = 4, parallel_chains = 4,
                 iter_warmup = 1000, iter_sampling = 1000, refresh = 0, show_exceptions = FALSE)
}

## Categorical variables

When N things are factors we often use dummy variables (0,1)'s to represent n-1 of them against a 'baseline' category. This is very standard practice, something I have done for years. However there are problems, because the baseline category has priority over the other variables in that by setting a baseline, we are *a priori* stating that we know more about the baseline category than the other categories. In some cases this may be true - I have typically used the category for which we have the most inoformation as the baseline - but it can also get cumbersome, as we need to assert a prior for each category. 

Instead, let's use an **index variable**, which will make things much simpler. First import the !Kung data:

In [ ]:
# Import data
kdata <- read.csv('howell.csv')
kdata$Sex <- c('Female', 'Male')[kdata$male + 1]
# Display top 5 rows
head(kdata, 5)

Next, you need some code to do the indexing, where each factor gets it's own number. The function below gives both a list of unique values, as well as an index number for each observation to connect them to the list of unique values. In R this is easy using a `factor`, which is R's built-in way of storing categorical data:

In [ ]:
# Helper functions
indexall <- function(L){
    poo <- unique(L)
    Ix <- as.integer(factor(L, levels = poo))
    list(poo, Ix)
}

With this, we can easily make an index for male/female in the !Kung data:

In [ ]:
tmp <- indexall(kdata$Sex)
Sex <- tmp[[1]]; Is <- tmp[[2]]
Sex; head(Is, 20)

Note that the categories are added on as they're encountered, so because the first entry is male, the male category is given the first index position, i.e. 1 (unlike Python, R and Stan both start counting at 1).

Next let's run two models, one using the dummy variable, and a second using the index variable. Note that in the Stan version we pass the prior SD for the male offset in as data, so we can change it later without re-compiling:

In [ ]:
dummy_code <- "
data {
  int<lower=0> N;
  vector[N] male;          // dummy variable (0/1)
  vector[N] height;
  real<lower=0> male_sd;   // prior SD for the male offset
}
parameters {
  real Female;                      // Baseline intercept
  real Male;                        // Male offset
  real<lower=0, upper=10> SD_obs;   // Error
}
model {
  Female ~ normal(178, 20);
  Male ~ normal(0, male_sd);
  SD_obs ~ uniform(0, 10);
  // Linear model and likelihood
  height ~ normal(Female + Male*male, SD_obs);
}
"
dummy <- cmdstan_model(write_stan_file(dummy_code))

In [ ]:
indexed_code <- "
data {
  int<lower=0> N;
  int<lower=1> K;                        // number of categories
  array[N] int<lower=1, upper=K> idx;    // index variable
  vector[N] height;
}
parameters {
  vector[K] Intercepts;
  real<lower=0, upper=10> SD_obs;
}
model {
  Intercepts ~ normal(178, 20);
  SD_obs ~ uniform(0, 10);
  // Linear model and likelihood
  height ~ normal(Intercepts[idx], SD_obs);
}
"
indexed <- cmdstan_model(write_stan_file(indexed_code))

In [ ]:
trace_d <- sample_stan(dummy, list(N = nrow(kdata), male = kdata$male, height = kdata$height, male_sd = 1))

In [ ]:
trace_i <- sample_stan(indexed, list(N = nrow(kdata), K = length(Sex), idx = Is, height = kdata$height))

In [ ]:
# Look at dummy model results
trace_d$summary()

In [ ]:
# Look at index-based results
tmp <- trace_i$summary()
tmp$Label <- c('lp__', Sex, 'Sd_obs')
tmp

In [ ]:
# Helper to overlay the posterior densities of two sets of draws
plot_density2 <- function(x1, x2, labels, main = "", xlim = NULL, vline = NULL){
    d1 <- density(x1); d2 <- density(x2)
    if (is.null(xlim)) xlim <- range(c(d1$x, d2$x))
    plot(d1, xlim = xlim, ylim = c(0, max(c(d1$y, d2$y))), main = main, xlab = "", col = "dodgerblue", lwd = 2)
    lines(d2, col = "orange", lwd = 2)
    if (!is.null(vline)) abline(v = vline)
    legend("topright", legend = labels, col = c("dodgerblue", "orange"), lwd = 2, bty = "n")
}

In [ ]:
compare_sexes <- function(trace_d, trace_i){
    options(repr.plot.width = 10, repr.plot.height = 4)
    par(mfrow = c(1, 2))
    female <- as.vector(trace_d$draws("Female"))
    male <- female + as.vector(trace_d$draws("Male"))
    ints <- trace_i$draws("Intercepts", format = "matrix")
    plot_density2(female, ints[, which(Sex == 'Female')], c('Dummy', 'Indexed'), main = 'Female',
                  xlim = c(130, 145), vline = mean(kdata$height[kdata$Sex == 'Female']))
    plot_density2(male, ints[, which(Sex == 'Male')], c('Dummy', 'Indexed'), main = 'Male',
                  xlim = c(130, 145), vline = mean(kdata$height[kdata$Sex == 'Male']))
    par(mfrow = c(1, 1))
}
compare_sexes(trace_d, trace_i)

So what has happened here? Well it turns out the $N(0,1)$ prior for males in the dummy version was too informative. So if we go for something wider:

In [ ]:
# Same compiled model, wider prior for the male offset
dummy_data <- list(N = nrow(kdata), male = kdata$male, height = kdata$height, male_sd = 5)

In [ ]:
trace_d <- sample_stan(dummy, dummy_data)

In [ ]:
compare_sexes(trace_d, trace_i)

So either version can get us to a simliar place, however the indexed version tends to be far easier to work with. 

# Selection distortion effects

Among the scariest aspects of statistical models is that things can become spurriously correlated due to the inclusion or exclusion of another, lurking variable. If we don't know about this, and how such things can be induced, we're doomed to report stuff that is complete malarky, adding noise to the scientific cannon. I'm sure I've done this at some point. But knowing is half the battle, so let's start with something called the selection-distortion effect. This is when a third, intervening variable is added that selects for a subset of the data and induces a spurroius correlation. This is something that is known as collider bias (we'll explain below), and it can be a big problem.

To illustrate, we can take a look at the simulated scientific distortion example on p162 in the book:


In [ ]:
# Set random number seed to get same answers
#set.seed(1914)

# Number of grant proposals
N <- 200
# Proportion to select
prop <- 0.1

# Uncorrelated Newsworthiness and trustworthiness scores
nw <- rnorm(N, 0, 1)
tw <- rnorm(N, 0, 1)

# Select top 10% of combined scores
score <- nw + tw
indx <- score > quantile(score, 1 - prop)

# How correlated?
cor_ <- cor(nw[indx], tw[indx])
cor_

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5)
plot(nw, tw, col = "dodgerblue", pch = 16, xlab = 'Newsworthiness', ylab = 'Trustworthiness')
# savefig('seldis0.jpg')

In [ ]:
plot(nw, tw, col = "dodgerblue", pch = 16, xlab = 'Newsworthiness', ylab = 'Trustworthiness')
points(nw[indx], tw[indx], col = 'red', pch = 16)
# savefig('seldis.jpg')

In most simulations, the red dots will have a negative correlation, entirely due to their having the highest total value for the two covariates. Provided we don't condition on score, this isn't a problem. If we did, we would induce a spurrious correlation that doesn't apply to the data as a whole. Tricky eh?

# DAGs and why to use 'em

Directed Acyclic Graphs are a really important development in modelling observational data (if you've done an experiment, then all power to you), because they give us some sort of ground to stand on in thinking about causality. Again, much of this creit goes to [Judea Pearl](https://en.wikipedia.org/wiki/Judea_Pearl), whose thinking around causality is nobel-prize worthy.

Pearl set out 4 key rules, that cover all the major bases. If you address these four things once you've made a causal model, causal inference can follow.

## The fork and the pipe

The fork and the pipe are equivalent mathematically, in that they are about conditing on an intermediating variable, but for the fork to break a spurrious correlation, and in the not conditioning to estimate an effect. We saw a fork example in the Southern divorce example, with a fork at marriage age (A) that, once conditioned on, breaks the association between marriage rate (M) and divorce (D).

For a pipe example, we can simulate some data relating to treatment effects on plants. The pipe is that the treatment (T) reduces fungus (F) and therfore aids in the growth of a plant (G).

In [ ]:
# Set number of plants
N <- 100

# Simulate initial heights
h0 <- rnorm(N, 10, 2)

# Assign treatments
treatment <- sample(c(0, 1), N, replace = TRUE)

# Simulate fungus conditional on treatment
beta_t <- 0.4
fungus <- rbinom(N, 1, .5 - beta_t*treatment)

# Generate end heights
h1 <- h0 + rnorm(N, 5 - 3*fungus, 1)

In [ ]:
brks <- pretty(h1, 15)
hist(h1[treatment == 0], breaks = brks, col = adjustcolor("dodgerblue", 0.7), main = "", xlab = 'Height (cm)')
hist(h1[treatment == 1], breaks = brks, col = adjustcolor("orange", 0.7), add = TRUE)
legend("topright", legend = c('control', 'treatment'), fill = adjustcolor(c("dodgerblue", "orange"), 0.7), bty = "n")
# savefig('treat.jpg')

You can see from what we've simulated that things are bimodial according to if the plants received the treatment or not. So, we have covariates for `initial height`, `treatment`, and `fungus` - to the multiple regression!

In [ ]:
plants_code <- "
data {
  int<lower=0> N;
  vector[N] h0;
  vector[N] treatment;
  vector[N] fungus;
  vector[N] h1;
}
parameters {
  real<lower=0> Intercept;   // Intercept
  real Treatment;            // Treatment effect
  real Fungus;               // Fungal effect
  real<lower=0> SD_obs;      // Error
}
model {
  Intercept ~ lognormal(0, 0.2);
  Treatment ~ normal(0, 0.5);
  Fungus ~ normal(0, 0.5);
  SD_obs ~ exponential(1);
  // Linear model
  vector[N] mu = (Intercept + Treatment*treatment + Fungus*fungus) .* h0;
  // Likelihood
  h1 ~ normal(mu, SD_obs);
}
"
plants <- cmdstan_model(write_stan_file(plants_code))
plant_data <- list(N = N, h0 = h0, treatment = treatment, fungus = fungus, h1 = h1)

In [ ]:
trace <- sample_stan(plants, plant_data)

In [ ]:
trace$summary()

In [ ]:
mcmc_intervals(trace$draws(c("Intercept", "Treatment", "Fungus", "SD_obs"))) + vline_0()
# ggsave('pipe.jpg', dpi = 300)

What the heck? We know the treatment will reduce the probability of fungus by 0.4, and that fungal-infested plants will grow an average of 3cm less (we made the data). So what's going on? Well, conditional on knowing that there is fungus, there is no benefit to knowing about the treatement, whereas conditional on knowing treatment it remains worth knowing if there is fungus (fungus happens to treated plants too). In this case fungus lies along the pipe between treatment and outcome, blocking information that would flow from treatment. Therefore if we want to estimate the effect of treatment we need a model without the post-treatment outcome, fungus:

In [ ]:
plants_t_code <- "
data {
  int<lower=0> N;
  vector[N] h0;
  vector[N] treatment;
  vector[N] h1;
}
parameters {
  real<lower=0> Intercept;           // Intercept
  real Treatment;                    // Treatment effect
  real<lower=0, upper=10> SD_obs;    // Error
}
model {
  Intercept ~ lognormal(0, 0.2);
  Treatment ~ normal(0, 0.5);
  SD_obs ~ uniform(0, 10);
  // Linear model
  vector[N] mu = (Intercept + Treatment*treatment) .* h0;
  // Likelihood
  h1 ~ normal(mu, SD_obs);
}
"
plants_t <- cmdstan_model(write_stan_file(plants_t_code))

In [ ]:
trace_t <- sample_stan(plants_t, list(N = N, h0 = h0, treatment = treatment, h1 = h1))

In [ ]:
trace_t$summary()

In [ ]:
mcmc_intervals(trace_t$draws(c("Intercept", "Treatment", "SD_obs"))) +
    vline_0() + geom_vline(xintercept = beta_t, colour = 'red')
# ggsave('tpipe.jpg', dpi = 300)

But now our effect is 0.15 (or so), not 0.4 - what's going on? Well, our known treatment effect of 0.4 is the reduction in the probability of fungus, while our treatment estimate above is on the effect of treatment on height. So we need to convert back to the probability scale, or convert our known number to the height effect scale. 

In [ ]:
# Average difference between full and fungal growth
avg_diff <- mean((h1 - h0)[fungus == 0]) - mean((h1 - h0)[fungus == 1])
avg_diff

In [ ]:
effect <- as.vector(trace_t$draws("Treatment"))*avg_diff
hist(effect, xlim = range(c(effect, beta_t)), main = "", xlab = 'p(fungus|Δheight)')
abline(v = beta_t, col = 'red')
# savefig('effect.jpg')

You can take a look at this model using the `dagitty` package in R, or online at [DAGitty.net](http://dagitty.net/dags.html)

In [ ]:
library(dagitty)
plant_dag <- dagitty("dag {
  H0 -> H1
  F -> H1
  T -> F
}")
coordinates(plant_dag) <- list(x = c(H0 = 0, H1 = 1, F = 2, T = 3), y = c(H0 = 0, H1 = 0, F = 0, T = 0))
plot(plant_dag)
# Which variables are conditionally independent in this DAG?
impliedConditionalIndependencies(plant_dag)

# Collider bias

McElreath has an agent-based model of happiness baked into his `rethinking` package, as outlined on p177, with five key conditions

    1. Each year, 20 people are born with uniformly distributed happiness values.
    2. Each year, each person ages one year. Happiness does not change.
    3. At age 18, individuals can become married. The odds of marriage each year are proportional to a person's happiness.
    4. Once married, individuals remain married.
    5. After age 65, individuals leave the sample (They move to Spain.)
    
We can import that simulated happiness data to work with it:

In [ ]:
# Happiness data
hdata <- read.csv('happiness.csv', row.names = 1)
head(hdata)

In [ ]:
plot_happy <- function(){
    options(repr.plot.width = 12, repr.plot.height = 5)
    plot(hdata$age[hdata$married == 0], hdata$happiness[hdata$married == 0], pch = 1,
         xlim = range(hdata$age), xlab = 'Age', ylab = 'Happiness')
    points(hdata$age[hdata$married == 1], hdata$happiness[hdata$married == 1], pch = 21, bg = "dodgerblue")
    legend("topleft", legend = c('Unmarried', 'Married'), pch = c(1, 21), pt.bg = c(NA, "dodgerblue"), bg = "white")
}
plot_happy()
# savefig('happy.jpg', width = 12)

The data above assumes happiness is uniformally distributed and never changes.

So, pretending we don't know anything about how this was generated, what sort of model should we build? 'Is age related to happiness' we might ask. And we should control for the effect of marriage, right?? A model representing these things would be

$$
\mu_i = \beta_{M[i]}+\beta_{A}A_i
$$

with indexed married/not-married intercepts ($\beta_{M[i]}$) and a parameter for changing happieness with age ($\beta_{A}$).

What makes for sensible priors sensible here? It's difficult to think about how much happiness should increase or decrease per year of age. First we can chuck out the kids, as they can't marry. Then if we scale the ages from 18 to 65 to be over the range 1 to 0, we know the range for happiness is -2 to 2, so we should be cover that range (i.e. 4 units) over the 0 to 1 interval of scaled ages. As 95% of the posterior mass is within 2SD of the mean, setting the prior SD to 4/2 will capture most of the range.

For the intercept, now equal to 0 at age 18, we can also span the range, using a $N(0,1)$ prior.


In Stan this would be:

In [ ]:
# Adults only
aindx <- hdata$age > 17
# Marriage index (1 = unmarried, 2 = married, since Stan counts from 1)
Im <- hdata$married[aindx] + 1
# Age - scaled so as to be between 0 and 1
AGE <- (hdata$age[aindx] - 18)/(65 - 18)
# Happiness
H <- hdata$happiness[aindx]

In [ ]:
happy_code <- "
data {
  int<lower=0> N;
  array[N] int<lower=1, upper=2> Im;   // marriage index
  vector[N] AGE;
  vector[N] H;
}
parameters {
  vector[2] Marriage;       // Intercepts
  real Age;                 // Age effect
  real<lower=0> SD_obs;     // Error
}
model {
  Marriage ~ normal(0, 1);
  Age ~ normal(0, 2);
  SD_obs ~ exponential(1);
  // Linear model and likelihood
  H ~ normal(Marriage[Im] + Age*AGE, SD_obs);
}
"
happy <- cmdstan_model(write_stan_file(happy_code))

In [ ]:
trace_h <- sample_stan(happy, list(N = length(H), Im = Im, AGE = AGE, H = H))

In [ ]:
trace_h$summary()

In [ ]:
mcmc_intervals(trace_h$draws(c("Marriage", "Age", "SD_obs"))) + vline_0()
# ggsave('happy1.jpg', dpi = 300)

Waaait a minute - we know that happiness is consistent across ages (everyone keeps their intial happiness levels in the simulation), so what the heck is the model doing? It is refelcting the model it was given, one where marriage acts as a collider to open a path between age and happiness. How does this occur? Well it occurs because marriage is a common consequence of both age and happiness - people who are older and happier are more likely to be married. Take a look again at the data: 

In [ ]:
plot_happy()

And you can see it - the older, happier people tend to be married. The collider. If instead of including it, we leave the collider path closed, we get the right result:

In [ ]:
happy2_code <- "
data {
  int<lower=0> N;
  vector[N] AGE;
  vector[N] H;
}
parameters {
  real Age;                 // Age effect
  real<lower=0> SD_obs;     // Error
}
model {
  Age ~ normal(0, 2);
  SD_obs ~ exponential(1);
  // Linear model and likelihood
  H ~ normal(Age*AGE, SD_obs);
}
"
happy2 <- cmdstan_model(write_stan_file(happy2_code))

In [ ]:
trace_h2 <- sample_stan(happy2, list(N = length(H), AGE = AGE, H = H))

In [ ]:
trace_h2$summary()

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5)
mcmc_intervals(trace_h2$draws(c("Age", "SD_obs"))) + vline_0()
# ggsave('happy2.jpg', dpi = 300)

# The Haunted DAG

Of all the many difficulties lurking in our statistical models, perhaps the scariest is that our estimates are influcenced by something we haven't measured, or even thought of. How can we deal with that? First let's look at the grandparents example. Here we are looking at the influence of both parent (P) and grandparent (G) education on children's education (C). The DAG for this is simple in the sense that $G\rightarrow P \rightarrow C$ (grandparents influence their kids, who influence their kids) AND $G\rightarrow C$ (grandparents influence their grandkids directly). 

![](GPC.jpg)

But what if we have a third, lurking variable $U$, that is a common influence on both parents and children? This could be something like the neighbourhood that the parents and kids live in, and by conditioning on the parents, we open up a backdoor path from $G$ to $C$ (via $U$)

![](GPCU.jpg)

What's messed up about this is that this can happen without our even knowing about $U$ or having ever observed it. Here's a simulation of how it works:

In [ ]:
# Number of families
N <- 200

# Direct effect of G on P
b_GP <- 1

# Direct effect of G on C
b_GC <- 0

# Direct effect of P on C
b_PC <- 1

# Direct effect of U on P and C
b_U <- 2

With these parameter estimates we can simulate from some random normals:

In [ ]:
# Simulate neighbourhood effects - some are positive (+1*U) some are negative (-1*U)
U <- 2*rbinom(N, 1, 0.5) - 1

# Simulate standard normal grandparents
G <- rnorm(N, 0, 1)

# Simulate Parents, conditional on their influeces (G and U)
P <- rnorm(N, b_GP*G + b_U*U, 1)

# Simulate Children, conditional on their influeces (G, P, and U)
C <- rnorm(N, b_PC*P + b_GC*G + b_U*U, 1)

Naievely running a regression model with parents and grandparents included, we get:

In [ ]:
edu_code <- "
data {
  int<lower=0> N;
  vector[N] G;
  vector[N] P;
  vector[N] C;
}
parameters {
  real Intercept;
  real Grandparents;        // Grandparent effect
  real Parents;             // Parent effect
  real<lower=0> SD_obs;     // Error
}
model {
  Intercept ~ normal(0, 1);
  Grandparents ~ normal(0, 1);
  Parents ~ normal(0, 1);
  SD_obs ~ exponential(1);
  // Linear model and likelihood
  C ~ normal(Intercept + Grandparents*G + Parents*P, SD_obs);
}
"
edu <- cmdstan_model(write_stan_file(edu_code))

In [ ]:
trace_e <- sample_stan(edu, list(N = N, G = G, P = P, C = C))

In [ ]:
trace_e$summary()

In [ ]:
mcmc_intervals(trace_e$draws(c("Intercept", "Grandparents", "Parents", "SD_obs"))) + vline_0()
# ggsave('pappy.jpg', dpi = 300)

So without knowing about the neighbourhood effect $U$, it seems like grandparents are messing up their grandkids!

From the DAG it's clear that conditioning on P opens up a backdoor path for grandparents to influence their grandkids indirectly through their kids choice of neighbourhood. 

To see this mechanically, take a look at the following figure:

In [ ]:
C_ <- stdize(C)
options(repr.plot.width = 6, repr.plot.height = 5)

plot(G[U == 1], C_[U == 1], col = 'royalblue', xlim = range(G), ylim = range(C_),
     xlab = 'Grandparent education (G)', ylab = 'Child education (C)')
points(G[U == -1], C_[U == -1], col = 'black')
legend("topleft", legend = c('N1', 'N2'), col = c('royalblue', 'black'), pch = 1, bty = "n")
# savefig('GC.jpg', width = 6)

First notice the positive relationship between G and C, even though the direct effect is 0 - how? Well $G\rightarrow P = 1$ and $P\rightarrow C = 1$ so there is a pipe from $G\rightarrow C$ evident in the plot. Now look what happens when we condition on parents - remember that this is a form of selection? - and highlight those parents who happen to lie between the 45th and 60th percentiles of education.

In [ ]:
# Flag parents that lie within percentile range
pmid <- P > quantile(P, 0.45) & P < quantile(P, 0.60)

# Plot all points
plot(G[U == 1], C_[U == 1], col = 'royalblue', xlim = range(G), ylim = range(C_),
     xlab = 'Grandparent education (G)', ylab = 'Child education (C)')
points(G[U == -1], C_[U == -1], col = 'black')

# Plot filled points
points(G[U == 1 & pmid], C_[U == 1 & pmid], pch = 16, col = 'royalblue')
points(G[U == -1 & pmid], C_[U == -1 & pmid], pch = 16, col = 'black')

legend("topleft", legend = c('N1', 'N2', 'N1|Pmid', 'N2|Pmid'), col = c('royalblue', 'black'),
       pch = c(1, 1, 16, 16), bty = "n")
# savefig('GC2.jpg', width = 6)

And there it is, hidden among the data - by conditioning on P, a negative association has been induced between grandparents and children that has entirely to do with the fact that the unmeasured neighbourhood effect is operating in the background, drawing down child scores in a hidden way. Spooky!

So what should we do about all this? Well if we had accounted for U:

In [ ]:
edu_new_code <- "
data {
  int<lower=0> N;
  vector[N] G;
  vector[N] P;
  vector[N] U;
  vector[N] C;
}
parameters {
  real Intercept;
  real Grandparents;        // Grandparent effect
  real Parents;             // Parent effect
  real Neighbourhood;       // Neighbourhood effect
  real<lower=0> SD_obs;     // Error
}
model {
  Intercept ~ normal(0, 1);
  Grandparents ~ normal(0, 1);
  Parents ~ normal(0, 1);
  Neighbourhood ~ normal(0, 1);
  SD_obs ~ exponential(1);
  // Linear model and likelihood
  C ~ normal(Intercept + Grandparents*G + Parents*P + Neighbourhood*U, SD_obs);
}
"
edu_new <- cmdstan_model(write_stan_file(edu_new_code))

In [ ]:
trace_n <- sample_stan(edu_new, list(N = N, G = G, P = P, U = U, C = C))

In [ ]:
trace_n$summary()

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5)
mcmc_intervals(trace_n$draws(c("Intercept", "Grandparents", "Parents", "Neighbourhood", "SD_obs"))) + vline_0()
# ggsave('pappyU.jpg', dpi = 300)

It gets it right. But this is kind of unsatisfactory, as we didn't know there was a problem, U stands for unmeasured after all.

For now, we'll remain haunted...